In [ ]:
from web3 import Web3
import json
import os

infura_key = '2e306bdddc7843108fe30334b2dfcfb2'
wallet_public_address = Web3.to_checksum_address('0x93753aa12E364f76a2bE0C22F1a4e4A443E715f1')
wallet_private_key = ''

web3 = Web3(Web3.HTTPProvider(f'https://sepolia.infura.io/v3/{infura_key}'))
print("Connected to Sepolia Testnet:", web3)
abi_file_path = os.path.join('./abis.json')
try:
    with open(abi_file_path, 'r', encoding='utf-8') as f:
        abi_data = json.load(f)
    print("ABI loaded successfully.")
except Exception as e:
    print(f"Error loading ABI: {e}")


factory_addr = '0x0227628f3F023bb0B980b67D528571c95c6DaC1c'
factory_contract = web3.eth.contract(factory_addr, abi=abi_data['UNISWAP_FACTORY_ABI'])
def get_pool_address(tokenA, tokenB, tier_fee, factory_contract):
    # Ensure tokens are in correct order (Uniswap V3 requires sorted token addresses)
    # tier_fee: 100 for 0.01%, 500 for 0.05%, 3000 for 0.3%, 10000 for 1%
    if tokenA > tokenB:
        tokenA, tokenB = tokenB, tokenA

    # Call the getPool function
    pool_address = factory_contract.functions.getPool(tokenA, tokenB, tier_fee).call()
    return pool_address

# approve the Universal Router to spend USDT on behalf of the wallet
universal_router_address = web3.to_checksum_address('0x3fC91A3afd70395Cd496C647d5a6CC9D4B2b7FAD')
universal_router_contract = web3.eth.contract(address=universal_router_address, abi=abi_data['UNIVERSAL_ROUTER_ABI'])
permit2_address = web3.to_checksum_address('0x000000000022D473030F116dDEE9F6B43aC78BA3')
permit2_contract = web3.eth.contract(address=permit2_address, abi=abi_data['PERMIT2_ABI'])

def send_tx(tx, private_key):
    signed_txn = web3.eth.account.sign_transaction(tx, private_key)
    raw_transaction = signed_txn.rawTransaction if hasattr(signed_txn, 'rawTransaction') else signed_txn.raw_transaction
    tx_hash = web3.eth.send_raw_transaction(raw_transaction)
    receipt = web3.eth.wait_for_transaction_receipt(tx_hash)
    print(f"Transaction sent: {tx_hash.hex()} ", end=" ; ")
    receipt_status = "success" if receipt.status == 1 else "failure"
    print(f"Transaction Status: {receipt_status}") 
    return tx_hash, receipt

def approve_token_spending(web3, contract, wallet_public_address, *approve_args):
    tx = contract.functions.approve(*approve_args).build_transaction({
        "from": wallet_public_address,
        "nonce": web3.eth.get_transaction_count(wallet_public_address),
    })
    tx_hash, receipt = send_tx(tx, wallet_private_key)

def get_current_price(tokenA, tokenB, fee, abi):
    pool_address = get_pool_address(tokenA, tokenB, fee, factory_contract)
    if pool_address == '0x0000000000000000000000000000000000000000':
        print("Pool does not exist for the given token pair and fee tier.")
        return None
    pool_contract = web3.eth.contract(address=pool_address, abi=abi)
    sqrtPriceX96 = pool_contract.functions.slot0().call()[0]
    price = (sqrtPriceX96 ** 2) / (2 ** 192)
    print(f"1 {tokenA} = {price} {tokenB}")
    return price

TOKEN_ADDRESSES = {
    "USTUSD": web3.to_checksum_address('0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c'),
    "USTETH": web3.to_checksum_address('0x4E22e9951770b98d4f3B2BA6647f0f40A15A4057'),
    "MiniDAI": web3.to_checksum_address('0xBB6d33A3f5E93DEd0e2F7fd19E77FdDDBd1d0B21'),
}

Connected to Sepolia Testnet: <web3.main.Web3 object at 0x000002AD4C5882E0>
ABI loaded successfully.


In [25]:
SWAP_IN_TOKEN_ADDRESS = TOKEN_ADDRESSES['USTUSD']
SWAP_OUT_TOKEN_ADDRESS = TOKEN_ADDRESSES['MiniDAI']
TIER_FEE = 100  # 0.01% fee tier

DECIMALS = 10**18
SWAP_AMOUNT = 350000 * DECIMALS  

In [26]:
print(f"Before swap, current price: ")
get_current_price(SWAP_IN_TOKEN_ADDRESS, SWAP_OUT_TOKEN_ADDRESS, TIER_FEE, abi_data['abi_pool'])

SWAP_IN_TOKEN_CONTRACT = web3.eth.contract(address=SWAP_IN_TOKEN_ADDRESS, abi=abi_data['ERC20_ABI'])
SWAP_OUT_TOKEN_CONTRACT = web3.eth.contract(address=SWAP_OUT_TOKEN_ADDRESS, abi=abi_data['ERC20_ABI'])
# approve_token_spending(web3, SWAP_IN_TOKEN_CONTRACT, wallet_public_address, permit2_address, 2**256 - 1)
# approve_token_spending(web3, SWAP_OUT_TOKEN_CONTRACT, wallet_public_address, permit2_address, 2**256 - 1)
# approve_token_spending(web3, permit2_contract, wallet_public_address, SWAP_IN_TOKEN_ADDRESS, universal_router_address, 2**160 - 1, 2**48 - 1)
# approve_token_spending(web3, permit2_contract, wallet_public_address, SWAP_OUT_TOKEN_ADDRESS, universal_router_address, 2**160 - 1, 2**48 - 1)

from uniswap_universal_router_decoder import FunctionRecipient, RouterCodec
codec = RouterCodec()
encoded_data = codec.encode.chain().v3_swap_exact_in(
        FunctionRecipient.SENDER,  # reciver
        SWAP_AMOUNT,  # amount in, (20000 USDT with 6 decimals)
        0,
        [   
            SWAP_IN_TOKEN_ADDRESS,
            TIER_FEE,
            SWAP_OUT_TOKEN_ADDRESS
        ],
    ).build(2**256 - 1)

tx_params = {
        "from": wallet_public_address,
        "to": universal_router_address,
        "gas": 500_000,
        "maxPriorityFeePerGas": web3.eth.max_priority_fee,
        "maxFeePerGas": 100 * 10**9,
        "type": '0x2',
        "chainId": 11155111,
        "nonce": web3.eth.get_transaction_count(wallet_public_address),
        "data": encoded_data,
        }

tx_hash, receipt = send_tx(tx_params, wallet_private_key)

print(f"After swap, new price: ")
get_current_price(SWAP_IN_TOKEN_ADDRESS, SWAP_OUT_TOKEN_ADDRESS, TIER_FEE, abi_data['abi_pool'])

    # if mode.lower() == "v3": # todo: need to calculate the payoff of V3
    #     payoff = y + p * x
    # elif mode.lower() == "v2": # todo: need to calculate the payoff of V2
    #     payoff = 2 * math.sqrt(k * p)
    # else:
    #     raise ValueError("mode must be v2 or v3")


Before swap, current price: 
1 0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c = 0.8113603636523941 0xBB6d33A3f5E93DEd0e2F7fd19E77FdDDBd1d0B21
Transaction sent: 8b901468396e5dfecec7ab40bf81c8bb7f0f8b94c409dd4b7b959696fede6050  ; Transaction Status: success
After swap, new price: 
1 0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c = 1.0183035614359355 0xBB6d33A3f5E93DEd0e2F7fd19E77FdDDBd1d0B21


1.0183035614359355